# Healthcare KG Population Intelligence — V5.1
### Clinical concept adjudication · provenance validation · guarded question answering · integration tests

**Input:** the actual `Healthcare_KG_V5_Outputs.zip` from Colab. **Output:** `Healthcare_KG_V5_1_Outputs.zip`.

This notebook never writes to GitHub or the baseline KG. Its curated links mean *conceptually related to a screening proxy*, **not** clinically confirmed diagnoses. All other V5 candidate links remain rejected or review-only. The output contains a full 88-link decision ledger. No external LLM or API keys required.

**Methodological restrictions:** single-visit CKD screen; probable T2D proxy; incomplete laboratory cross-cycle harmonization; missingness uncertainty; no Kerendia eligibility or time-trend conclusions. The underlying V4.2 epidemiological review status is carried forward unchanged.

In [ ]:
import io, json, zipfile, hashlib, re, csv, math, shutil, tempfile
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
OUT=Path('/content/Healthcare_KG_V5_1_Outputs') if Path('/content').exists() else Path.cwd()/'Healthcare_KG_V5_1_Outputs'
OUT.mkdir(parents=True,exist_ok=True)
ZIP_NAME='Healthcare_KG_V5_Outputs.zip'
if not Path(ZIP_NAME).exists():
    try:
        from google.colab import files
        print('Upload the V5 output ZIP (containing baseline_kg_COPY.json).')
        uploaded=files.upload()
        ZIP_NAME=next(k for k in uploaded if k.lower().endswith('.zip'))
    except ImportError:
        raise FileNotFoundError('Place V5 output ZIP in the notebook working directory.')
archive=Path(ZIP_NAME)
z=zipfile.ZipFile(archive)
required={'baseline_kg_COPY.json','population_observations_v5.json','population_clinical_links_v5.json','v5_validation_report.json','v4_2_corrected_population_estimates.csv','v4_2_validation_report.json','v4_2_cdc_methodology_audit.json'}
missing=required-set(z.namelist());assert not missing, f'Missing files: {missing}'
print('Input:',archive,'entries:',len(z.namelist()))

## 1. Immutable baseline and input manifest

In [ ]:
baseline_bytes=z.read('baseline_kg_COPY.json')
sha=hashlib.sha256(baseline_bytes).hexdigest()
kg=json.loads(baseline_bytes)
original_nodes={str(x[0]):{'type':str(x[1]),'description':str(x[2])} for x in kg['nodes']}
original_edges=kg['edges']
prev=json.loads(z.read('v5_validation_report.json'))
assert sha==prev['baseline_sha256'], 'Baseline checksum mismatch'
assert len(original_nodes)==prev['baseline_nodes'] and len(original_edges)==prev['baseline_edges']
obs=json.loads(z.read('population_observations_v5.json'))
links=json.loads(z.read('population_clinical_links_v5.json'))
assert len({o['id'] for o in obs})==len(obs)
assert all(x['clinical_node_id'] in original_nodes for x in links)
assert all(x['observation_id'] in {o['id'] for o in obs} for x in links)
manifest=[{'filename':name,'bytes':len(z.read(name)),'sha256':hashlib.sha256(z.read(name)).hexdigest()} for name in sorted(z.namelist())]
(OUT/'input_manifest_v5_1.json').write_text(json.dumps({'source_zip_sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'files':manifest},indent=2))
print('Baseline:',len(original_nodes),'nodes,',len(original_edges),'edges; SHA256:',sha)

## 2. Explicit clinical concept adjudication

Only broad, directly identifiable CKD and T2D concept nodes receive **curated conceptual links**. Links to complications, stages, comorbidities or broad syndromes are not treated as equivalent to the screening proxy. The policy is intentionally conservative. Review the complete ledger before promoting additional mappings.

In [ ]:
# Allowlist is tied to exact node descriptions in the supplied baseline.
CURATED={
 'D1':('probable T2D screening concept','Type 2 diabetes mellitus'),
 'D18':('probable T2D screening concept','Type 2 diabetes'),
 'D19':('CKD screening concept','Chronic kidney disease'),
 'D40':('CKD screening concept','Chronic kidney disease'),
}
# Validate the allowlist against the actual baseline to avoid approving a reused ID.
for nid,(role,term) in CURATED.items():
    assert nid in original_nodes, f'Missing curated node {nid}'
    assert original_nodes[nid]['type'].lower()=='disease', f'Unexpected type for {nid}'
    assert term.lower() in original_nodes[nid]['description'].lower(), f'Changed description for {nid}'
ledger=[]
for x in links:
    nid=x['clinical_node_id'];role=x['concept_role']
    accepted=(nid in CURATED and CURATED[nid][0]==role)
    row={**x,
        'clinical_node_type':original_nodes[nid]['type'],
        'clinical_node_description':original_nodes[nid]['description'],
        'adjudication':'CURATED_CONCEPTUAL_ONLY' if accepted else 'REJECTED_AS_NON_EQUIVALENT',
        'adjudication_reason':('Explicit broad disease-concept match; does not assert patient-level diagnosis or eligibility' if accepted else 'Lexical mention alone is insufficient; may describe stage, complication, risk, syndrome or a narrower population'),
        'clinical_confirmation':False,
        'reviewer':'rule_based_v5_1',
        'human_review_completed':False}
    ledger.append(row)
accepted=[x for x in ledger if x['adjudication']=='CURATED_CONCEPTUAL_ONLY']
assert len(ledger)==len(links)
(OUT/'clinical_link_adjudication_v5_1.json').write_text(json.dumps(ledger,indent=2))
pd.DataFrame(ledger).to_csv(OUT/'clinical_link_adjudication_v5_1.csv',index=False)
print('Candidate links:',len(ledger),'curated conceptual:',len(accepted),'rejected:',len(ledger)-len(accepted))
display(pd.DataFrame(ledger)['adjudication'].value_counts().rename_axis('decision').reset_index(name='count'))

## 3. Provenance checks and reliability labels

In [ ]:
est=pd.read_csv(io.BytesIO(z.read('v4_2_corrected_population_estimates.csv')))
source_report=json.loads(z.read('v4_2_validation_report.json'))
audit=json.loads(z.read('v4_2_cdc_methodology_audit.json'))
checks=[]
def check(label,condition,details=''):
    checks.append({'check':label,'passed':bool(condition),'details':details})
check('baseline_checksum',sha==prev['baseline_sha256'])
check('clinical_links_referential_integrity',len(ledger)==len(links))
check('observations_have_unique_ids',len(obs)==len({o['id'] for o in obs}))
for o in obs:
    rows=est[(est['cycle'].astype(str)==str(o['period']))&(est['scenario'].astype(str)==str(o['scenario']))]
    ok=len(rows)==1
    if ok:
        r=rows.iloc[0]
        ok=math.isclose(float(o['estimated_adults']),float(r['weighted_cases']),rel_tol=1e-8) and math.isclose(float(o['weighted_denominator']),float(r['weighted_denominator']),rel_tol=1e-8)
        ok=ok and math.isclose(float(o['count_ci_95'][0]),float(r['count_ci_low']),rel_tol=1e-8) and math.isclose(float(o['count_ci_95'][1]),float(r['count_ci_high']),rel_tol=1e-8)
        ok=ok and str(o['survey_weight'])==str(r['weight'])
    check('observation_matches_v4_2_'+o['id'],ok)
    check('ci_contains_estimate_'+o['id'],float(o['count_ci_95'][0])<=float(o['estimated_adults'])<=float(o['count_ci_95'][1]))
check('2021_2023_uses_phlebotomy_weight',all(o['survey_weight']=='WTPH2YR' for o in obs if o['period']=='2021-2023'))
check('clinical_claims_blocked',all(not o.get('treatment_eligibility_estimate',True) and not o.get('trend_comparison_allowed',True) for o in obs))
check('epidemiology_review_carried_forward',all(o['epidemiological_validation']=='REVIEW_REQUIRED' for o in obs))
for o in obs:
    o['reliability_v5_1']={
       'computational':'CROSSCHECK_REPORTED_PASS_NOT_REEXECUTED',
       'epidemiological':'REVIEW_REQUIRED',
       'clinical':'SCREENING_PROXY_ONLY',
       'small_cell_warning':bool(o['unweighted_cases']<30),
       'trend_claims_allowed':False,
       'treatment_eligibility_claims_allowed':False,
       'human_concept_review_completed':False}
(OUT/'population_observations_v5_1.json').write_text(json.dumps(obs,indent=2))
(OUT/'provenance_checks_v5_1.json').write_text(json.dumps(checks,indent=2))
assert all(c['passed'] for c in checks),[c for c in checks if not c['passed']]
print('Provenance checks passed:',len(checks))

## 4. Separate KG observation layer and guarded question-answering

This is a deterministic evidence-aware query prototype. It does not use an LLM to infer eligibility, temporal trends, or causal relationships.

In [ ]:
layer={'schema_version':'5.1','baseline_sha256':sha,
       'observations':obs,'curated_conceptual_links':accepted,
       'rejected_link_count':len(ledger)-len(accepted),
       'note':'Conceptual screening-proxy links only. No clinical diagnosis or treatment eligibility inferred.'}
(OUT/'population_kg_layer_v5_1.json').write_text(json.dumps(layer,indent=2))
def answer(question,period='2021-2023',scenario='primary'):
    q=question.lower()
    forbidden_eligibility=('eligible','eligibility','indicated','prescribe','prescription','addressable market','tam','sam','revenue','sales','kerendia patients')
    forbidden_trends=('trend','increase over time','decline over time','changed over time','growing','growth rate','year over year','compare cycles','across cycles')
    if any(t in q for t in forbidden_eligibility):
        return {'status':'BLOCKED_UNSUPPORTED_CLINICAL_INFERENCE','answer':'NHANES single-visit CKD + probable T2D screening observations do not establish treatment eligibility or commercial addressable market.','evidence_ids':[]}
    if any(t in q for t in forbidden_trends):
        return {'status':'BLOCKED_UNVALIDATED_TREND','answer':'Cross-cycle trend claims are blocked pending serum creatinine laboratory comparability and epidemiological validation.','evidence_ids':[]}
    if not any(t in q for t in ('ckd','kidney','diabet','nhanes','albumin')):
        return {'status':'OUT_OF_SCOPE','answer':'No validated observation is available for this question.','evidence_ids':[]}
    matches=[o for o in obs if o['period']==period and o['scenario']==scenario]
    if len(matches)!=1:return {'status':'NO_OBSERVATION','answer':'No matching period and scenario found.','evidence_ids':[]}
    o=matches[0]
    return {'status':'SCREENING_ESTIMATE_WITH_CAVEATS',
        'answer':f"NHANES {period} ({scenario}) estimates {o['estimated_adults']:,.0f} U.S. adults (95% CI {o['count_ci_95'][0]:,.0f}–{o['count_ci_95'][1]:,.0f}) meeting a single-visit CKD laboratory screen and probable T2D proxy. Complete-case screening prevalence: {o['screening_prevalence_complete_case']*100:.2f}%. This is not confirmed CKD, T2D, Kerendia eligibility, or a validated trend.",
        'evidence_ids':[o['id']], 'weight':o['survey_weight'],
        'validation':'EPIDEMIOLOGICAL_REVIEW_REQUIRED',
        'source':'CDC NHANES, derived V4.2 outputs',
        'definition':o['definition']}
questions=['How many adults have CKD and diabetes?','How many are eligible for Kerendia?','What is the CKD trend across cycles?','What is the population of Mars?']
examples=[{'question':q,**answer(q)} for q in questions]
(OUT/'guarded_query_examples_v5_1.json').write_text(json.dumps(examples,indent=2))
for e in examples:print(e['question'],'->',e['status'])

## 5. Automated integration tests and export

In [ ]:
tests={
 'baseline_unmodified_in_memory':hashlib.sha256(json.dumps(kg,ensure_ascii=False).encode()).hexdigest()!='' and hashlib.sha256(baseline_bytes).hexdigest()==sha,
 'all_input_observations_retained':len(obs)==4,
 'all_candidate_links_adjudicated':len(ledger)==len(links),
 'no_curated_link_implies_diagnosis':all(not l['clinical_confirmation'] for l in accepted),
 'no_curated_link_has_human_approval':all(not l['human_review_completed'] for l in accepted),
 'eligibility_guard':answer('How many patients are eligible for Kerendia?')['status']=='BLOCKED_UNSUPPORTED_CLINICAL_INFERENCE',
 'trend_guard':answer('What is the CKD trend?')['status']=='BLOCKED_UNVALIDATED_TREND',
 'population_answer_has_provenance':bool(answer('How many adults have CKD and diabetes?')['evidence_ids']),
 'unrelated_question_not_fabricated':answer('How far is the moon?')['status']=='OUT_OF_SCOPE',
 'primary_and_bridge_are_separate':len([o for o in obs if o['period']=='2021-2023'])==2,
 'all_provenance_checks_pass':all(x['passed'] for x in checks),
 'baseline_source_still_identical':hashlib.sha256(z.read('baseline_kg_COPY.json')).hexdigest()==sha,
}
assert all(tests.values()),{k:v for k,v in tests.items() if not v}
report={'version':'V5.1','generated_utc':datetime.now(timezone.utc).isoformat(),
        'input_archive_sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),
        'baseline_sha256':sha,'baseline_nodes':len(original_nodes),'baseline_edges':len(original_edges),
        'observations':len(obs),'candidate_links':len(links),'curated_conceptual_links':len(accepted),
        'rejected_links':len(ledger)-len(accepted), 'human_review_completed':False,
        'tests':tests,'test_pass_count':sum(tests.values()),
        'source_epidemiological_status':'REVIEW_REQUIRED',
        'remaining_blockers':['Human review of curated clinical mappings','Serum creatinine cross-cycle comparability','External epidemiological validation','Missing-data model assumptions','Clinical eligibility cannot be derived from these observations'],
        'github_accessed':False,'baseline_modified':False,
        'status':'INTEGRATION_TESTS_PASS_CLINICAL_REVIEW_REQUIRED'}
(OUT/'v5_1_validation_report.json').write_text(json.dumps(report,indent=2))
# Include exact baseline copy as a separate, unchanged file for checksum verification.
(OUT/'baseline_kg_COPY.json').write_bytes(baseline_bytes)
(OUT/'README.txt').write_text('V5.1 review-only population KG layer. Clinical links are conceptual and not human-approved. No GitHub modifications. See validation report and full link adjudication ledger.\n')
zip_path=OUT.parent/'Healthcare_KG_V5_1_Outputs.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zz:
    for f in sorted(OUT.iterdir()):
        if f.is_file():zz.write(f,arcname=f.name)
print('PASS:',len(tests),'integration tests. ZIP:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:pass